# Validación Bronze — steam_games_review

Este notebook perfila `steam_game_intelligence.bronze.steam_games_review` antes de diseñar sus tablas Silver y DLQ. El campo `reviews` se trata como texto opaco: no se asume una estructura interna hasta observar los resultados.

Las consultas son de solo lectura. La última celda exporta los resultados a Markdown.

## 1. Volumen, claves y metadatos

In [ ]:
%sql
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT app_id) AS distinct_app_ids,
       COUNT_IF(app_id IS NULL) AS null_app_id_rows,
       COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows,
       COUNT_IF(reviews IS NULL) AS null_reviews_rows,
       COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews_rows,
       COUNT(DISTINCT _ingested_at) AS ingestion_timestamps,
       MIN(_ingested_at) AS first_ingested_at,
       MAX(_ingested_at) AS last_ingested_at
FROM steam_game_intelligence.bronze.steam_games_review;

## 2. Nulos e inconsistencias de columnas

In [ ]:
%sql
SELECT column_name, null_count
FROM (
  SELECT COUNT_IF(app_id IS NULL) AS app_id,
         COUNT_IF(name IS NULL) AS name,
         COUNT_IF(reviews IS NULL) AS reviews,
         COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
         COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.steam_games_review
) UNPIVOT (null_count FOR column_name IN (app_id, name, reviews, _ingested_at, _source))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Duplicados y snapshots

In [ ]:
%sql
SELECT app_id, CAST(_ingested_at AS DATE) AS ingestion_date,
       COUNT(*) AS row_count,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT reviews) AS distinct_reviews
FROM steam_game_intelligence.bronze.steam_games_review
GROUP BY app_id, CAST(_ingested_at AS DATE)
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 200;

In [ ]:
%sql
SELECT app_id, COUNT(*) AS snapshot_rows,
       COUNT(DISTINCT CAST(_ingested_at AS DATE)) AS ingestion_days,
       COUNT(DISTINCT reviews) AS distinct_reviews,
       MIN(_ingested_at) AS first_seen,
       MAX(_ingested_at) AS last_seen
FROM steam_game_intelligence.bronze.steam_games_review
WHERE app_id IS NOT NULL
GROUP BY app_id
HAVING COUNT(*) > 1
ORDER BY snapshot_rows DESC
LIMIT 200;

## 4. Perfil del campo `reviews`

Se mide la longitud, la diversidad y el primer carácter del contenido. Esto permite decidir si Silver debe conservar el valor como texto o si existe evidencia suficiente para parsearlo.

In [ ]:
%sql
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT reviews) AS distinct_reviews,
       MIN(LENGTH(reviews)) AS min_length,
       percentile_approx(LENGTH(reviews), 0.5) AS median_length,
       MAX(LENGTH(reviews)) AS max_length,
       COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) RLIKE '^[\\[{]') AS structured_prefix_rows,
       COUNT_IF(reviews IS NOT NULL AND NOT (TRIM(reviews) RLIKE '^[\[{]')) AS other_prefix_rows
FROM steam_game_intelligence.bronze.steam_games_review;

In [ ]:
%sql
SELECT CASE WHEN reviews IS NULL THEN 'NULL'
            WHEN TRIM(reviews) = '' THEN 'BLANK'
            ELSE SUBSTRING(TRIM(reviews), 1, 1) END AS first_character,
       COUNT(*) AS row_count,
       MIN(LENGTH(reviews)) AS min_length,
       MAX(LENGTH(reviews)) AS max_length
FROM steam_game_intelligence.bronze.steam_games_review
GROUP BY CASE WHEN reviews IS NULL THEN 'NULL'
            WHEN TRIM(reviews) = '' THEN 'BLANK'
            ELSE SUBSTRING(TRIM(reviews), 1, 1) END
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
SELECT app_id, name, LENGTH(reviews) AS reviews_length, reviews,
       _ingested_at, _source
FROM steam_game_intelligence.bronze.steam_games_review
WHERE reviews IS NOT NULL AND TRIM(reviews) <> ''
ORDER BY reviews_length DESC, app_id
LIMIT 50;

## 5. Registros candidatos a rechazo

Estas filas se muestran para revisar posibles reglas de Silver/DLQ. Los conteos del diagnóstico no constituyen todavía reglas aprobadas.

In [ ]:
%sql
SELECT app_id, name, reviews, _ingested_at, _source,
       filter(array(
         CASE WHEN app_id IS NULL OR app_id <= 0 THEN 'app_id_invalido' END,
         CASE WHEN name IS NULL OR TRIM(name) = '' THEN 'name_vacio' END,
         CASE WHEN reviews IS NULL OR TRIM(reviews) = '' THEN 'reviews_vacio' END,
         CASE WHEN _ingested_at IS NULL THEN '_ingested_at_nulo' END
       ), reason -> reason IS NOT NULL) AS candidate_reasons
FROM steam_game_intelligence.bronze.steam_games_review
WHERE app_id IS NULL OR app_id <= 0
   OR name IS NULL OR TRIM(name) = ''
   OR reviews IS NULL OR TRIM(reviews) = ''
   OR _ingested_at IS NULL
LIMIT 200;

In [ ]:
%python
from datetime import datetime
import os

table_name = 'steam_game_intelligence.bronze.steam_games_review'
notebook_name = '05_validacion_steam_games_review'
queries = [
    ('Volumen y cobertura', "SELECT COUNT(*) AS total_rows, COUNT(DISTINCT app_id) AS distinct_app_ids, COUNT_IF(app_id IS NULL) AS null_app_id_rows, COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows, COUNT_IF(reviews IS NULL) AS null_reviews_rows, COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews_rows, COUNT(DISTINCT _ingested_at) AS ingestion_timestamps, MIN(_ingested_at) AS first_ingested_at, MAX(_ingested_at) AS last_ingested_at FROM steam_game_intelligence.bronze.steam_games_review"),
    ('Nulos por columna', "SELECT column_name, null_count FROM (SELECT COUNT_IF(app_id IS NULL) AS app_id, COUNT_IF(name IS NULL) AS name, COUNT_IF(reviews IS NULL) AS reviews, COUNT_IF(_ingested_at IS NULL) AS _ingested_at, COUNT_IF(_source IS NULL) AS _source FROM steam_game_intelligence.bronze.steam_games_review) UNPIVOT (null_count FOR column_name IN (app_id, name, reviews, _ingested_at, _source)) WHERE null_count > 0 ORDER BY null_count DESC"),
    ('Duplicados por app y día', "SELECT app_id, CAST(_ingested_at AS DATE) AS ingestion_date, COUNT(*) AS row_count, COUNT(DISTINCT name) AS distinct_names, COUNT(DISTINCT reviews) AS distinct_reviews FROM steam_game_intelligence.bronze.steam_games_review GROUP BY app_id, CAST(_ingested_at AS DATE) HAVING COUNT(*) > 1 ORDER BY row_count DESC LIMIT 200"),
    ('Historial de snapshots', "SELECT app_id, COUNT(*) AS snapshot_rows, COUNT(DISTINCT CAST(_ingested_at AS DATE)) AS ingestion_days, COUNT(DISTINCT reviews) AS distinct_reviews, MIN(_ingested_at) AS first_seen, MAX(_ingested_at) AS last_seen FROM steam_game_intelligence.bronze.steam_games_review WHERE app_id IS NOT NULL GROUP BY app_id HAVING COUNT(*) > 1 ORDER BY snapshot_rows DESC LIMIT 200"),
    ('Perfil de reviews', "SELECT COUNT(*) AS total_rows, COUNT(DISTINCT reviews) AS distinct_reviews, MIN(LENGTH(reviews)) AS min_length, percentile_approx(LENGTH(reviews), 0.5) AS median_length, MAX(LENGTH(reviews)) AS max_length, COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) RLIKE '^[\\[{]') AS structured_prefix_rows, COUNT_IF(reviews IS NOT NULL AND NOT (TRIM(reviews) RLIKE '^[\\[{]') )) AS other_prefix_rows FROM steam_game_intelligence.bronze.steam_games_review"),
    ('Distribución del prefijo', "SELECT CASE WHEN reviews IS NULL THEN 'NULL' WHEN TRIM(reviews) = '' THEN 'BLANK' ELSE SUBSTRING(TRIM(reviews), 1, 1) END AS first_character, COUNT(*) AS row_count, MIN(LENGTH(reviews)) AS min_length, MAX(LENGTH(reviews)) AS max_length FROM steam_game_intelligence.bronze.steam_games_review GROUP BY CASE WHEN reviews IS NULL THEN 'NULL' WHEN TRIM(reviews) = '' THEN 'BLANK' ELSE SUBSTRING(TRIM(reviews), 1, 1) END ORDER BY row_count DESC LIMIT 100"),
    ('Muestras de reviews', "SELECT app_id, name, LENGTH(reviews) AS reviews_length, reviews, _ingested_at, _source FROM steam_game_intelligence.bronze.steam_games_review WHERE reviews IS NOT NULL AND TRIM(reviews) <> '' ORDER BY reviews_length DESC, app_id LIMIT 50"),
    ('Candidatos a rechazo', "SELECT app_id, name, reviews, _ingested_at, _source FROM steam_game_intelligence.bronze.steam_games_review WHERE app_id IS NULL OR app_id <= 0 OR name IS NULL OR TRIM(name) = '' OR reviews IS NULL OR TRIM(reviews) = '' OR _ingested_at IS NULL LIMIT 200"),
    ('Muestra de estructura', f'SELECT * FROM {table_name} LIMIT 1'),
]
def markdown_value(value):
    if value is None:
        return 'NULL'
    return str(value).replace('|', '\\|').replace('\r', ' ').replace('\n', ' ')
report = [f'# Resultados de validación - {notebook_name}', '', f'- Tabla: `{table_name}`', f'- Ejecutado: `{datetime.now().astimezone().isoformat()}`', '', 'Las consultas son diagnósticas y no fijan por sí solas las reglas finales de Silver/DLQ.', '', '## Consultas y resultados']
for title, query in queries:
    rows = [row.asDict(recursive=True) for row in spark.sql(query).limit(200).collect()]
    report.extend(['', f'### {title}', f'_Filas exportadas: {len(rows)}; máximo aplicado: 200._', ''])
    if not rows:
        report.append('No se devolvieron filas.')
        continue
    columns = list(rows[0].keys())
    report.append('| ' + ' | '.join(columns) + ' |')
    report.append('| ' + ' | '.join(['---'] * len(columns)) + ' |')
    for row in rows:
        report.append('| ' + ' | '.join(markdown_value(row.get(column)) for column in columns) + ' |')
context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = '/notebooks/data_validations'
if marker not in context_path:
    raise RuntimeError(f'No se pudo determinar la raíz del repositorio desde: {context_path}')
repo_path = context_path.split(marker, 1)[0]
workspace_repo_path = repo_path if repo_path.startswith('/Workspace') else '/Workspace' + repo_path
output_dir = os.path.join(workspace_repo_path, 'notebooks', 'data_validations', 'result_outputs')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f'resultados_{notebook_name}.md')
temp_path = output_path + '.tmp'
with open(temp_path, 'w', encoding='utf-8', newline='\n') as handle:
    handle.write('\n'.join(report) + '\n')
os.replace(temp_path, output_path)
print(f'Markdown generado: {output_path}; consultas exportadas: {len(queries)}')
